# 03 — Relationships, Cardinality & Join Coverage

**Objetivo:** validar empíricamente las relaciones entre las siete fuentes antes de materializar joins.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Métricas
- **Coverage:** proporción de filas del lado izquierdo con match.
- **Multiplicity:** cuántas filas del lado derecho puede tener una key.
- **Expansion factor:** filas esperadas tras LEFT JOIN / filas originales.

Un join que ejecuta sin error no es necesariamente un join correcto.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import (
    discover_dataset_files,
    load_registry,
    load_relationship_candidates,
)

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
RELATIONSHIP_CANDIDATES = load_relationship_candidates(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}


{'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'sanctions': 'SECOPII_-_Multas_y_Sanciones_20260921.csv',
 'payments': 'SECOP_II_-_Plan_de_pagos_20260921.csv'}

In [2]:
from argos.common.reports import write_report
from argos.profiling.relationships import (
    analyze_all_relationships,
    analyze_relationship_candidates,
    select_relationship_candidate,
)

cfg = PROJECT_CONFIG["profiling"]
gate_cfg = PROJECT_CONFIG["m1_gate"]

# Resolve contracts -> processes empirically instead of hard-coding the failed
# Proceso de Compra -> ID del Proceso hypothesis.
candidate_spec = RELATIONSHIP_CANDIDATES["contracts_processes"]
relationship_candidates = analyze_relationship_candidates(
    key_prefix=candidate_spec.key,
    left_dataset=candidate_spec.left_dataset,
    right_dataset=candidate_spec.right_dataset,
    left_keys=candidate_spec.left_keys,
    right_key_candidates=candidate_spec.right_key_candidates,
    files=FILES,
    sample_size=cfg["csv_sample_size"],
    min_coverage=gate_cfg["relationship_candidate_min_coverage"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

selected_contracts_processes, relationship_resolution = select_relationship_candidate(
    relationship_candidates,
    relationship_key="contracts_processes",
    min_coverage=gate_cfg["relationship_candidate_min_coverage"],
)

write_report(
    relationship_candidates,
    paths.reports / "profiling" / "relationship_candidate_analysis.parquet",
)
write_report(
    relationship_resolution,
    paths.reports / "profiling" / "relationship_resolution.parquet",
)

relationship_candidates[[
    "candidate_right_keys",
    "status",
    "left_row_coverage",
    "max_right_multiplicity",
    "expansion_factor",
    "requires_right_canonicalization",
]]


,candidate_right_keys,status,left_row_coverage,max_right_multiplicity,expansion_factor,requires_right_canonicalization
0,[id_del_portafolio],analyzed,0.975490,8840,2.050849,True
1,[referencia_del_proceso],analyzed,0.000002,1,1.000000,False
2,[id_del_proceso],analyzed,0.000000,0,1.000000,False


In [3]:
from argos.profiling.grain import analyze_duplicate_structure

# Characterize the exact process key selected by the relationship resolver.
# This is the grain evidence Silver will use before canonicalizing Processes.
selected_process_key = tuple(selected_contracts_processes.right_keys)
selected_process_grain = pd.DataFrame([
    analyze_duplicate_structure(
        "processes",
        FILES["processes"],
        selected_process_key,
        sample_size=cfg["csv_sample_size"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
])

grain_path = paths.reports / "profiling" / "grain_resolution.parquet"
existing_grain = pd.read_parquet(grain_path) if grain_path.exists() else pd.DataFrame()
selected_key_label = "+".join(selected_process_key)
if not existing_grain.empty:
    existing_grain = existing_grain.loc[
        ~((existing_grain["dataset"] == "processes") & (existing_grain["key"] == selected_key_label))
    ]
grain_resolution = pd.concat([existing_grain, selected_process_grain], ignore_index=True)
write_report(grain_resolution, grain_path)

# Re-run the relationship audit with the empirically selected contracts->processes key.
resolved_relationships = dict(RELATIONSHIPS)
resolved_relationships["contracts_processes"] = selected_contracts_processes

join_analysis = analyze_all_relationships(
    resolved_relationships,
    FILES,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    join_analysis,
    paths.reports / "profiling" / "join_analysis.parquet",
)

display(relationship_resolution)
display(selected_process_grain)
join_analysis[[
    "relationship", "status", "left_dataset", "right_dataset",
    "left_keys", "right_keys", "left_row_coverage",
    "max_right_multiplicity", "expansion_factor",
]]


,relationship,status,left_dataset,right_dataset,left_keys,selected_right_keys,left_row_coverage,max_right_multiplicity,expansion_factor,requires_right_canonicalization,selection_rule,minimum_required_coverage
0,contracts_processes,resolved,contracts,processes,[proceso_de_compra],[id_del_portafolio],0.97549,8840,2.050849,True,max_coverage_then_min_expansion,0.9


,dataset,key,raw_columns,status,distinct_nonnull_keys,duplicate_key_count,rows_on_duplicate_keys,duplicate_excess_rows,exact_duplicate_excess_rows,exact_duplicate_key_count,conflicting_duplicate_key_count,max_key_multiplicity,max_payload_variants,grain_interpretation,silver_implication
0,processes,id_del_portafolio,[ID del Portafolio],analyzed,8540484,222287,886226,663939,136838,41186,181101,8840,7420,multiple_payload_variants_per_key,canonicalize_or_model_versioned_rows


,relationship,status,left_dataset,right_dataset,left_keys,right_keys,left_row_coverage,max_right_multiplicity,expansion_factor
0,contracts_suppliers,analyzed,contracts,suppliers,[codigo_proveedor],[codigo],0.946647,1.0,1.000000
1,bidders_processes,analyzed,bidders,processes,[id_procedimiento],[id_del_proceso],0.988318,8840.0,4.565023
2,bidders_suppliers,analyzed,bidders,suppliers,[codigo_proveedor],[codigo],0.000000,0.0,1.000000
3,additions_contracts,analyzed,additions,contracts,[id_contrato],[id_contrato],0.999946,1.0,1.000000
4,payments_contracts,missing_columns,payments,contracts,NaN,NaN,NaN,NaN,NaN
5,sanctions_contracts,analyzed,sanctions,contracts,[id_contrato],[id_contrato],0.831522,1.0,1.000000
6,contracts_processes,analyzed,contracts,processes,[proceso_de_compra],[id_del_portafolio],0.975490,8840.0,2.050849


## Criterio de salida
`docs/data-model.md` debe distinguir **relaciones verificadas** de **hipótesis todavía no aceptadas**.
No construyas una mega-tabla en Bronze.

In [4]:
# Review, do not automatically reject, relations with high raw multiplicity.
# Processes is known to be non-unique in Bronze; Silver must canonicalize it
# before materializing one-to-one/many-to-one joins.
join_analysis.loc[
    (join_analysis["status"] != "analyzed")
    | (join_analysis["left_row_coverage"].fillna(0) < 0.90)
    | (join_analysis["expansion_factor"].fillna(1) > 1.05)
]


,relationship,left_dataset,right_dataset,left_keys,right_keys,left_raw_columns,right_raw_columns,status,left_rows,right_rows,left_rows_null_key,right_rows_null_key,matched_left_rows,left_row_coverage,matched_distinct_left_keys,max_right_multiplicity,expected_left_join_rows,expansion_factor,missing_left,missing_right,ambiguous_left,ambiguous_right
1,bidders_processes,bidders,processes,[id_procedimiento],[id_del_proceso],[ID Procedimiento],[ID del Proceso],analyzed,2316965.0,9204423.0,0.0,0.0,2289898.0,0.988318,681939.0,8840.0,10576999.0,4.565023,NaN,NaN,NaN,NaN
2,bidders_suppliers,bidders,suppliers,[codigo_proveedor],[codigo],[Codigo Proveedor],[Codigo],analyzed,2316965.0,1612550.0,0.0,0.0,0.0,0.000000,0.0,0.0,2316965.0,1.000000,NaN,NaN,NaN,NaN
4,payments_contracts,payments,contracts,NaN,NaN,NaN,NaN,missing_columns,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[id_contrato],[],NaN,NaN
5,sanctions_contracts,sanctions,contracts,[id_contrato],[id_contrato],[ID Contrato],[ID Contrato],analyzed,552.0,6066730.0,0.0,0.0,459.0,0.831522,405.0,1.0,552.0,1.000000,NaN,NaN,NaN,NaN
6,contracts_processes,contracts,processes,[proceso_de_compra],[id_del_portafolio],[Proceso de Compra],[ID del Portafolio],analyzed,6066730.0,9204423.0,0.0,0.0,5918035.0,0.975490,5210349.0,8840.0,12441945.0,2.050849,NaN,NaN,NaN,NaN
